# 04. Prepare for Deployment

Load artifacts, wrap inference, validate, export to MLflow, and document deployment.


In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.metrics import classification_report, accuracy_score, f1_score


In [ ]:
models_dir = os.path.join("..", "models")
proc_dir = os.path.join("..", "data", "processed")

with open(os.path.join(models_dir, "best_model_metadata.json"), encoding="utf-8") as f:
    metadata = json.load(f)
with open(os.path.join(proc_dir, "feature_names.json"), encoding="utf-8") as f:
    feature_names = json.load(f)
with open(os.path.join(proc_dir, "label_classes.json"), encoding="utf-8") as f:
    label_classes = json.load(f)

with open(metadata["model_path"], "rb") as f:
    clf = pickle.load(f)
with open(os.path.join(models_dir, "scaler.pkl"), "rb") as f:
    scaler = pickle.load(f)

X_test = np.load(os.path.join(proc_dir, "X_test.npy"))
y_test = np.load(os.path.join(proc_dir, "y_test.npy"))
print("Loaded model:", metadata["best_model_name"])
print("Features:", len(feature_names))


In [ ]:
class InferencePipeline:
    """Preprocess raw feature vectors then predict (classifier + scaler)."""

    def __init__(self, scaler, model, feature_names, class_names):
        self.scaler = scaler
        self.model = model
        self.feature_names = list(feature_names)
        self.class_names = list(class_names)

    def preprocess(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        return self.scaler.transform(X)

    def predict(self, X_raw):
        Xs = self.preprocess(X_raw)
        return self.model.predict(Xs)

    def predict_proba(self, X_raw):
        Xs = self.preprocess(X_raw)
        if hasattr(self.model, "predict_proba"):
            return self.model.predict_proba(Xs)
        raise AttributeError("Model does not support predict_proba")

pipeline = InferencePipeline(scaler, clf, feature_names, label_classes)


In [ ]:
# Three sample inputs (raw feature order matches feature_names)
samples = [
    [5.1, 3.5, 1.4, 0.2, 1.4 * 0.2, 5.1 * 3.5, (1.4 * 0.2) / (5.1 * 3.5 + 1e-8)],  # ~setosa-like
    [6.0, 2.9, 4.5, 1.5, 4.5 * 1.5, 6.0 * 2.9, (4.5 * 1.5) / (6.0 * 2.9 + 1e-8)],
    [6.5, 3.0, 5.2, 2.0, 5.2 * 2.0, 6.5 * 3.0, (5.2 * 2.0) / (6.5 * 3.0 + 1e-8)],
]
for i, row in enumerate(samples):
    pred = pipeline.predict(row)[0]
    print(f"Sample {i}: predicted class index = {pred} ({label_classes[pred]})")
    if hasattr(clf, "predict_proba"):
        print("  proba:", pipeline.predict_proba(row)[0])


In [ ]:
y_pred = pipeline.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 macro:", f1_score(y_test, y_pred, average="macro"))
print("\n", classification_report(y_test, y_pred, target_names=label_classes))


In [ ]:
# Log pipeline to MLflow and register in Model Registry (server at http://localhost:5000)
mlflow.set_tracking_uri("http://localhost:5000")
registered_name = "iris_classifier"
with mlflow.start_run(run_name="register-inference-pipeline"):
    mlflow.sklearn.log_model(pipeline, artifact_path="inference_pipeline")
    run_uri = f"runs:/{mlflow.active_run().info.run_id}/inference_pipeline"
mv = mlflow.register_model(run_uri, registered_name)
print("Registered:", registered_name, "version", mv.version)


In [ ]:
deploy_dir = os.path.join(models_dir, "deployment")
os.makedirs(deploy_dir, exist_ok=True)
pipeline_path = os.path.join(deploy_dir, "inference_pipeline.pkl")
with open(pipeline_path, "wb") as f:
    pickle.dump(pipeline, f)

deployment_info = {
    "model_name": registered_name,
    "model_version": mv.version,
    "mlflow_tracking_uri": "http://localhost:5000",
    "pipeline_path": pipeline_path,
    "scaler_path": os.path.join(models_dir, "scaler.pkl"),
    "feature_names": feature_names,
    "label_classes": label_classes,
    "best_model_metadata": metadata,
}
info_path = os.path.join(deploy_dir, "deployment_info.json")
with open(info_path, "w", encoding="utf-8") as f:
    json.dump(deployment_info, f, indent=2)
print("Saved:", pipeline_path, info_path)


## API request examples

**JSON body (features in order of `feature_names`):**

```json
{
  "features": [5.1, 3.5, 1.4, 0.2, 0.28, 17.85, 0.0157]
}
```

**`curl` (example):**

```bash
curl -s -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [5.1, 3.5, 1.4, 0.2, 0.28, 17.85, 0.0157]}'
```

Adjust host/port/path to match your serving app (e.g. FastAPI in `api.py`).


## Deployment checklist

- [ ] MLflow tracking server reachable at `http://localhost:5000` (or set env `MLFLOW_TRACKING_URI`).
- [ ] `best_model.pkl`, `scaler.pkl`, and processed `.npy` / JSON artifacts present under `models/` and `data/processed/`.
- [ ] Inference container image built and scanned; secrets not baked into images.
- [ ] Health and readiness probes configured for the inference service.
- [ ] Load tests for expected QPS; autoscaling limits (CPU/memory) set in Kubernetes/Helm if used.
- [ ] Monitoring: request latency, error rate, and model drift (e.g. Evidently) wired to dashboards.
- [ ] Rollback plan: previous Model Registry version tagged and deployable.
